In [2]:
#imports

from tldhamburg.backend.DataFetcher import DataFetcher
import shapely

In [3]:
#collect primary_signals

fetcher = DataFetcher()

endpoint = "Datastreams"
params = {
    "$filter" : "properties/layerName eq 'primary_signal'"
}

fetcher.fetch_data(endpoint,params)

signals = fetcher.get_data()

In [ ]:
#define coordinates (9,53)
coordinates = [9.5321, 53.34339]

#iterate over error margin until only one mathing Datastream is found
error = 1
lower_bound = 0
upper_bound = 1
while True:
    all_matched_Datastreams_ids = list()
    reference_point = shapely.geometry.Point(coordinates[0], coordinates[1])
    for entry in signals:
        if not "observedArea" in entry:
            continue
        geoJSON_observedArea = shapely.geometry.shape(entry["observedArea"])
        distance = reference_point.distance(geoJSON_observedArea)
        if distance < error:
            if not "@iot.id" in entry:
                print("whoops")
                break
            datastream_id = entry["@iot.id"]
            all_matched_Datastreams_ids.append(datastream_id)

    if len(all_matched_Datastreams_ids) == 1:
        print(f"#Datastreams found = {len(all_matched_Datastreams_ids)}\n{all_matched_Datastreams_ids} with distance:{error}")
        break

    if len(all_matched_Datastreams_ids) > 1:
        upper_bound = error

    if len(all_matched_Datastreams_ids) < 1:
        lower_bound = error
    
    #coordinates are inside observedArea of multiple Datastreams
    if upper_bound - lower_bound < 1e-10:
        print(f"#Datastreams found = {len(all_matched_Datastreams_ids)}\n{all_matched_Datastreams_ids} with distance:{error}")
        break
    error = (upper_bound + lower_bound) / 2





#DS = 1
[39905]
